In [1]:
import os
import json
import pickle
import random
import numpy as np
import pandas as pd

# Setup

In [3]:
system_message = """You are a helpful assistant designed to output JSON.

# Background
Given a soft query containing a free variable f, there are four candidate entities for this variable and you need to choose the best of them to satisfy the soft query most. In order to do so, you can first compute the confidence value to see whether the chosen candidate entity satisfies the soft query after substituting the free variable with a given candidate entity. After computing the confidence values for each corresponding candidate entity, you need to pick the entity that leads to the highest confidence value. The detailed steps are described below:

## Definition of Soft Atomic Constraint:
A soft atomic constraint c, a.k.a. a soft atomic formula or its negation, is in the form of (h, r, t, \alpha, \beta) or \neg (h, r, t, \alpha, \beta).

### Notation Description:
In each constraint, we have four different types of variables.
1. r is a relation; 
2. h and t are two terms. Each term represents an entity or a variable whose values are entities. And free variable is a term.
3. \alpha is called the necessity value, which represents the minimal requirement of the uncertainty degree of this constraint. It can be any decimal between 0.0 and 1.0. If the confidence value is less than the necessity value, the constraint is not satisfied, and thus the final confidence value becomes negative infinity; 
4. \beta represents the priority of this constraint and can be any decimal between 0.0 and 1.0.

### Confidence Value of Soft Constraints V(c):
1. The triple (h,r,t) comes from the relation fact in the knowledge graph. In our setting, the relation fact r(h,t) is not boolean, and it has a confidence value in the range [0.0,1.0] according to its plausibility. When the constraint is negative, you need to first estimate r(h,t)—the confidence value of r(h,t)—and use 1-r(h,t) as the final confidence value for this negative constraint.   
2. \alpha is the threshold in our filter function, working as follows:  
f(v, \alpha) = v,           if  v \geq \alpha,
                    	         -\infty,    if  v < \alpha.
3. \beta is a coefficient.

Thus, the final equation becomes:
V(h, r, t, \alpha, \beta) = \beta \times f(r(h,t), \alpha),
V(\neg (h, r, t, \alpha, \beta)) = \beta \times f(1-r(h,t), \alpha).

## Definition of Conjunctive Queries \phi:
Soft conjunctive queries are composed of soft constraints.

### Notation Description:
Conjunctive query \phi = c_1 \land \dots \land c_n, where c_i is a soft constraint.

### Confidence Value of Soft Conjunctive Queries V(\phi) :
First, you need to compute the confidence values of all soft constraints in this soft conjunctive query. Then, you can simply sum up these confidence values as the final confidence value of the soft conjunctive query as follows:
V(\phi) = \sum_i V(c_i).

# Output Format
Please output your response in the JSON format, where the first element is the index of the best candidate entity among the four options, and the second element is your explanation for your choice.
"""

output_instruction = """Please return the best candidate for f1 to satisfy the above soft query most.
"""

from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

# Load selected questions

In [4]:
type0000_queries = pd.read_excel(open('Data Annotation v2.xlsx', 'rb'), sheet_name='type0000')  
type0000_queries = type0000_queries[type0000_queries["Is meaningful query?"] == 1.0]
print(type0000_queries.shape)
type0000_queries

FileNotFoundError: [Errno 2] No such file or directory: 'Data Annotation v2.xlsx'

In [26]:
# type0000_queries.iloc[0]["query"]

'Soft Query: (lime, is related to, f1, 0.3146, 0.4)\nFour candidate entities: imagination, citrus, green, lemon\n'

In [ ]:
# ask_gpt(type0000_queries.iloc[0]["query"])

In [32]:
def ask_gpt(question):
    response = client.chat.completions.create(
        # model="gpt-3.5-turbo",
        model="gpt-3.5-turbo-1106",
        response_format={ "type": "json_object" },
          messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": question + output_instruction}
          ]
    )
    
    return response.choices[0].message.content

type0000_first_ten = type0000_queries[:10].copy()
type0000_first_ten["gpt_response"] = type0000_first_ten["query"].map(ask_gpt)
type0000_first_ten


,Unnamed: 0,raw,solution,query,Is meaningful query?,FWZ: answer,DY: answer,3: answer,4: answer,Unnamed: 9,Unnamed: 10,Stats,Value,gpt_response
1,1,"(0, 'r1(s1,f1)', {'r1': 0, 's1': 331, 'a1': 0....",lemon,"Soft Query: (lime, is related to, f1, 0.3146, ...",1.0,NaN,lemon,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate_index"": 2,\n ""explanatio..."
2,2,"(1, 'r1(s1,f1)', {'r1': 3, 's1': 4458, 'a1': 0...",distillery,"Soft Query: (still, is synonym of, f1, 0.8927,...",1.0,NaN,distillery,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate_index"": 3,\n ""explanatio..."
4,4,"(3, 'r1(s1,f1)', {'r1': 2, 's1': 3500, 'a1': 0...",game,"Soft Query: (sport, is a, f1, 0.8927, 0.7)\nFo...",1.0,NaN,game,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate_index"": 3,\n ""explanatio..."
6,6,"(5, 'r1(s1,f1)', {'r1': 9, 's1': 1771, 'a1': 0...",capacious,"Soft Query: (large, is similar to, f1, 0.8927,...",1.0,NaN,capacious,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""index"": 2,\n ""explanation"": ""The confid..."
7,7,"(6, 'r1(s1,f1)', {'r1': 0, 's1': 3173, 'a1': 0...",certificate,"Soft Query: (degree, is related to, f1, 0.7093...",1.0,NaN,certificate,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate_index"": 2,\n ""explanatio..."
8,8,"(7, 'r1(s1,f1)', {'r1': 16, 's1': 701, 'a1': 0...",buy car,"Soft Query: (people, is capable of, f1, 0.7093...",1.0,NaN,count,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate_index"": 3,\n ""explanatio..."
9,9,"(8, 'r1(s1,f1)', {'r1': 0, 's1': 2364, 'a1': 0...",street,"Soft Query: (main, is related to, f1, 0.3146, ...",1.0,NaN,street,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate"": ""portion"",\n ""explanat..."
10,10,"(9, 'r1(s1,f1)', {'r1': 27, 's1': 45, 'a1': 0....",itch,"Soft Query: (person, does not desire, f1, 0.89...",1.0,NaN,imprisonment,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""index"": 2,\n ""explanation"": ""The entity..."
11,11,"(10, 'r1(s1,f1)', {'r1': 0, 's1': 1307, 'a1': ...",days,"Soft Query: (year, is related to, f1, 0.3146, ...",1.0,NaN,time,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""index"": 1,\n ""explanation"": ""The best c..."
12,12,"(11, 'r1(s1,f1)', {'r1': 0, 's1': 2852, 'a1': ...",camera,"Soft Query: (picture, is related to, f1, 0.314...",1.0,NaN,camera,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate"": {\n ""index"": 3,\n ..."


In [33]:
type0002_queries = pd.read_excel(open('Data Annotation v2.xlsx', 'rb'), sheet_name='type0002')  
type0002_queries = type0002_queries[type0002_queries["Is meaningful query?"] == 1.0]
print(type0002_queries.shape)
type0002_queries

(22, 13)


,Unnamed: 0,raw,solution,query,Is meaningful query?,FWZ: answer,DY: answer,3: answer,4: answer,Unnamed: 9,Unnamed: 10,Stats,Value
1,1,"(0, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2':...",food,"Soft Query: (bread, is related to, f1, 0.7093,...",1.0,food,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,"(8, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2':...",asking,"Soft Query: (question, is related to, f1, 0.31...",1.0,asking,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,24,"(23, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",houses,"Soft Query: (room, is related to, f1, 0.3146, ...",1.0,building,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,28,"(27, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",train,"Soft Query: (station, is related to, f1, 0.314...",1.0,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31,31,"(30, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",making,"Soft Query: (writing, is related to, f1, 0.314...",1.0,writing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,36,"(35, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",school,"Soft Query: (class, is related to, f1, 0.7093,...",1.0,school,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38,38,"(37, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",small,"Soft Query: (baby, is related to, f1, 0.7093, ...",1.0,kid,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43,43,"(42, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",people,"Soft Query: (men, is related to, f1, 0.7093, 0...",1.0,people,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47,47,"(46, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",tiny,"Soft Query: (drop, is related to, f1, 0.7093, ...",1.0,tiny,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52,52,"(51, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",action,"Soft Query: (dance, is related to, f1, 0.7093,...",1.0,action,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
type0002_first_ten = type0002_queries[:10].copy()
type0002_first_ten["gpt_response"] = type0002_first_ten["query"].map(ask_gpt)
type0002_first_ten

,Unnamed: 0,raw,solution,query,Is meaningful query?,FWZ: answer,DY: answer,3: answer,4: answer,Unnamed: 9,Unnamed: 10,Stats,Value,gpt_response
1,1,"(0, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2':...",food,"Soft Query: (bread, is related to, f1, 0.7093,...",1.0,food,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate_index"": 3,\n ""explanatio..."
9,9,"(8, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2':...",asking,"Soft Query: (question, is related to, f1, 0.31...",1.0,asking,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""chosen_candidate_index"": 3,\n ""explanat..."
24,24,"(23, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",houses,"Soft Query: (room, is related to, f1, 0.3146, ...",1.0,building,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""index_of_best_candidate"": 2,\n ""explana..."
28,28,"(27, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",train,"Soft Query: (station, is related to, f1, 0.314...",1.0,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate_index"": 4,\n ""explanatio..."
31,31,"(30, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",making,"Soft Query: (writing, is related to, f1, 0.314...",1.0,writing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate"": ""art"",\n ""explanation""..."
36,36,"(35, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",school,"Soft Query: (class, is related to, f1, 0.7093,...",1.0,school,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""chosen_candidate"": ""learning"",\n ""expla..."
38,38,"(37, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",small,"Soft Query: (baby, is related to, f1, 0.7093, ...",1.0,kid,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate"": ""kid"",\n ""explanation""..."
43,43,"(42, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",people,"Soft Query: (men, is related to, f1, 0.7093, 0...",1.0,people,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""index_of_best_candidate"": 2,\n ""explana..."
47,47,"(46, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",tiny,"Soft Query: (drop, is related to, f1, 0.7093, ...",1.0,tiny,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""index_of_best_candidate"": 3,\n ""explana..."
52,52,"(51, '(r1(s1,f1))&(r2(s2,f1))', {'r1': 0, 'r2'...",action,"Soft Query: (dance, is related to, f1, 0.7093,...",1.0,action,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""best_candidate_index"": 3,\n ""explanatio..."


In [36]:
with pd.ExcelWriter('test_ten_queries.xlsx', engine='xlsxwriter') as writer:
    type0000_first_ten.to_excel(writer, sheet_name='type0000')
    type0002_first_ten.to_excel(writer, sheet_name='type0002')

    workbook  = writer.book
    cell_format = workbook.add_format({'text_wrap': True})

    worksheet = writer.sheets["type0000"]
    worksheet.set_column('A:Z', cell_format=cell_format)
    worksheet = writer.sheets["type0002"]
    worksheet.set_column('A:Z', cell_format=cell_format)